In [2]:
import numpy as np
import pandas as pd

In [3]:
data = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")


In [4]:
data

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,0,Jansport,Leather,Medium,7.0,Yes,No,Tote,Black,11.611723,112.15875
1,1,Jansport,Canvas,Small,10.0,Yes,Yes,Messenger,Green,27.078537,68.88056
2,2,Under Armour,Leather,Small,2.0,Yes,No,Messenger,Red,16.643760,39.17320
3,3,Nike,Nylon,Small,8.0,Yes,No,Messenger,Green,12.937220,80.60793
4,4,Adidas,Canvas,Medium,1.0,Yes,Yes,Messenger,Green,17.749338,86.02312
...,...,...,...,...,...,...,...,...,...,...,...
299995,299995,Adidas,Leather,Small,9.0,No,No,Tote,Blue,12.730812,129.99749
299996,299996,Jansport,Leather,Large,6.0,No,Yes,Tote,Blue,26.633182,19.85819
299997,299997,Puma,Canvas,Large,9.0,Yes,Yes,Backpack,Pink,11.898250,111.41364
299998,299998,Adidas,Nylon,Small,1.0,No,Yes,Tote,Pink,6.175738,115.89080


In [5]:
data.describe()

,id,Compartments,Weight Capacity (kg),Price
count,300000.000000,300000.000000,299862.000000,300000.000000
mean,149999.500000,5.443590,18.029994,81.411107
std,86602.684716,2.890766,6.966914,39.039340
min,0.000000,1.000000,5.000000,15.000000
25%,74999.750000,3.000000,12.097867,47.384620
50%,149999.500000,5.000000,18.068614,80.956120
75%,224999.250000,8.000000,24.002375,115.018160
max,299999.000000,10.000000,30.000000,150.000000


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    300000 non-null  int64  
 1   Brand                 290295 non-null  object 
 2   Material              291653 non-null  object 
 3   Size                  293405 non-null  object 
 4   Compartments          300000 non-null  float64
 5   Laptop Compartment    292556 non-null  object 
 6   Waterproof            292950 non-null  object 
 7   Style                 292030 non-null  object 
 8   Color                 290050 non-null  object 
 9   Weight Capacity (kg)  299862 non-null  float64
 10  Price                 300000 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 25.2+ MB


In [7]:
object_cols = [col for col in data.columns if data[col].dtype == 'object']

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in object_cols:
    data[col] = le.fit_transform(data[col])
    test[col] = le.transform(test[col])

In [9]:
data

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,0,1,1,1,7.0,1,0,2,0,11.611723,112.15875
1,1,1,0,2,10.0,1,1,1,3,27.078537,68.88056
2,2,4,1,2,2.0,1,0,1,5,16.643760,39.17320
3,3,2,2,2,8.0,1,0,1,3,12.937220,80.60793
4,4,0,0,1,1.0,1,1,1,3,17.749338,86.02312
...,...,...,...,...,...,...,...,...,...,...,...
299995,299995,0,1,2,9.0,0,0,2,1,12.730812,129.99749
299996,299996,1,1,0,6.0,0,1,2,1,26.633182,19.85819
299997,299997,3,0,0,9.0,1,1,0,4,11.898250,111.41364
299998,299998,0,2,2,1.0,0,1,2,4,6.175738,115.89080


In [10]:
data.isna().sum()

id                        0
Brand                     0
Material                  0
Size                      0
Compartments              0
Laptop Compartment        0
Waterproof                0
Style                     0
Color                     0
Weight Capacity (kg)    138
Price                     0
dtype: int64

In [11]:
data.dropna(inplace=True)

In [12]:
X = data.drop(["id", "Price"], axis=1)
y = data['Price']

In [18]:
X_test = test.drop(["id"], axis=1)
X_test.fillna(0, inplace=True)

In [19]:
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import numpy as np

# Assuming X and y are already defined
# X -> Features, y -> Target

# Train XGBoost
dtrain = xgb.DMatrix(X, label=y)
dtest = xgb.DMatrix(X_test)
xgb_params = {'objective': 'reg:squarederror', 'learning_rate': 0.1, 'max_depth': 5}
xgb_model = xgb.train(xgb_params, dtrain)

# Train CatBoost
cat_model = cb.CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=5, verbose=0)
cat_model.fit(X, y)

# Train LightGBM
lgb_train = lgb.Dataset(X, label=y)
lgb_params = {'objective': 'regression', 'learning_rate': 0.1, 'max_depth': 5, 'verbose': -1}
lgb_model = lgb.train(lgb_params, lgb_train)

# Making Predictions
xgb_pred = xgb_model.predict(dtest)
cat_pred = cat_model.predict(X_test)
lgb_pred = lgb_model.predict(X_test)

# Final Prediction: Average of all three models
final_pred = (xgb_pred + cat_pred + lgb_pred) / 3

# Print the first few predictions
print(final_pred[:10])

[81.22880617 81.51178764 84.14847789 81.37927964 77.18531967 81.61425661
 82.42756802 82.8817504  84.08509209 81.07288044]


In [14]:
xgb_model.get_score()

{'Brand': 43.0,
 'Material': 29.0,
 'Size': 31.0,
 'Compartments': 26.0,
 'Laptop Compartment': 17.0,
 'Waterproof': 22.0,
 'Style': 21.0,
 'Color': 24.0,
 'Weight Capacity (kg)': 96.0}

In [20]:
sub = pd.read_csv("data/sample_submission.csv")
sub

,id,Price
0,300000,81.411
1,300001,81.411
2,300002,81.411
3,300003,81.411
4,300004,81.411
...,...,...
199995,499995,81.411
199996,499996,81.411
199997,499997,81.411
199998,499998,81.411


In [21]:
sub['Price'] = final_pred

In [22]:
sub.to_csv("outputs/xcl_equal.csv", index=False)